In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import h5py
import json
plt.style.use('./graph_preset.mplstyle')

In [2]:
read_path = Path("./0320104525_gp10/results.h5")

In [3]:
with h5py.File(read_path, "r") as f: # read_paths[#] that you want to read
    print(f"--- Structure of {read_path} ---")

    def print_structure(name, obj):
        # データセットの場合は形状とデータ型も表示
        if isinstance(obj, h5py.Dataset):
            print(f"  {name} (Dataset) | Shape: {obj.shape}, Dtype: {obj.dtype}")
        # グループの場合はグループ名を表示
        elif isinstance(obj, h5py.Group):
            print(f"  {name} (Group)")

    f.visititems(print_structure)
    print("---------------------------------")

--- Structure of 0320104525_gp10\results.h5 ---
  input (Group)
  learning_curve (Group)
  output (Group)
  output/repeat_1 (Dataset) | Shape: (50, 12), Dtype: float32
  output/repeat_10 (Dataset) | Shape: (50, 12), Dtype: float32
  output/repeat_2 (Dataset) | Shape: (50, 12), Dtype: float32
  output/repeat_3 (Dataset) | Shape: (50, 12), Dtype: float32
  output/repeat_4 (Dataset) | Shape: (50, 12), Dtype: float32
  output/repeat_5 (Dataset) | Shape: (50, 12), Dtype: float32
  output/repeat_6 (Dataset) | Shape: (50, 12), Dtype: float32
  output/repeat_7 (Dataset) | Shape: (50, 12), Dtype: float32
  output/repeat_8 (Dataset) | Shape: (50, 12), Dtype: float32
  output/repeat_9 (Dataset) | Shape: (50, 12), Dtype: float32
---------------------------------


In [4]:
df_data = dict()

def store_dataset(name, obj):
    if isinstance(obj, h5py.Dataset):
        print(f"  Loading: {name} | Shape: {obj.shape}")
        df = pd.DataFrame(obj[:])
        df_data[name] = df

def store_dataset(name, obj):
    if isinstance(obj, h5py.Dataset):
        print(f"  Loading: {name} | Shape: {obj.shape}")

        # columns 属性があれば JSON から復元
        cols_attr = obj.attrs.get("columns", None)
        columns = None
        if cols_attr is not None:
            # 古い h5py だと bytes / np.bytes_ で返ることがある
            if isinstance(cols_attr, (bytes, np.bytes_)):
                cols_attr = cols_attr.decode("utf-8")
            columns = json.loads(cols_attr)

        # DataFrame 化（列名があれば使う）
        data = obj[:]  # dset[:, :] と同じ
        if columns is not None:
            df = pd.DataFrame(data, columns=columns)
        else:
            df = pd.DataFrame(data)

        df_data[name] = df


with h5py.File(read_path, "r") as f:
    print(f"--- Loading all datasets from {read_path} ---")
    f.visititems(store_dataset)
    print("---------------------------------------------")

print("\n--- Dictionary Keys ---")
print(list(df_data.keys()))
print("-----------------------")

--- Loading all datasets from 0320104525_gp10\results.h5 ---
  Loading: output/repeat_1 | Shape: (50, 12)
  Loading: output/repeat_10 | Shape: (50, 12)
  Loading: output/repeat_2 | Shape: (50, 12)
  Loading: output/repeat_3 | Shape: (50, 12)
  Loading: output/repeat_4 | Shape: (50, 12)
  Loading: output/repeat_5 | Shape: (50, 12)
  Loading: output/repeat_6 | Shape: (50, 12)
  Loading: output/repeat_7 | Shape: (50, 12)
  Loading: output/repeat_8 | Shape: (50, 12)
  Loading: output/repeat_9 | Shape: (50, 12)
---------------------------------------------

--- Dictionary Keys ---
['output/repeat_1', 'output/repeat_10', 'output/repeat_2', 'output/repeat_3', 'output/repeat_4', 'output/repeat_5', 'output/repeat_6', 'output/repeat_7', 'output/repeat_8', 'output/repeat_9']
-----------------------


In [5]:
pd.set_option('display.max_rows', None)

In [6]:
df_data["output/repeat_1"]

,h1,h2,h3,h4,h5,a,b,k,S11,Metric,gamma,best
0,2.549437,1.376073,3.914615,3.534132,1.022744,2.458794,7.549732,5.408100,-6.143101,NaN,NaN,-6.143101
1,2.807047,3.976305,2.929660,3.335714,3.504282,7.998098,4.567379,2.264445,-5.737009,NaN,NaN,-6.143101
2,3.459691,2.402525,3.484578,3.122299,2.390082,5.100726,3.299805,1.317214,-6.665107,NaN,NaN,-6.665107
3,3.995099,4.667475,1.634205,2.118672,4.205350,3.676328,3.065019,4.274950,-10.750782,NaN,NaN,-10.750782
4,1.313422,2.610496,1.202148,4.796302,4.763370,5.621225,6.010160,3.661078,-10.027316,NaN,NaN,-10.750782
5,3.742285,1.626122,4.092422,4.678951,3.859453,7.579437,3.915360,5.813602,-3.690381,NaN,NaN,-10.750782
6,4.916859,3.215982,1.284287,1.912749,1.487046,6.002345,2.169673,2.602978,-5.427535,NaN,NaN,-10.750782
7,3.071559,4.921653,2.325199,2.728871,1.944321,5.355998,7.023681,4.488497,-8.697922,NaN,NaN,-10.750782
8,1.720516,3.786389,3.189129,2.378715,2.087115,7.177397,5.776182,4.925746,-7.579873,NaN,NaN,-10.750782
9,4.393016,2.924479,2.020439,1.666849,2.837135,2.354068,4.956446,3.956184,-12.912526,NaN,NaN,-12.912526


In [8]:
combined = pd.concat(df_data, names=["source"])
min_pos = combined["S11"].idxmin()

print(f"Min S11: {combined.loc[min_pos, 'S11']}")
print(f"Location: {min_pos}")
print("That row:")
print(combined.loc[min_pos])

Min S11: -28.42096710205078
Location: ('output/repeat_8', 44)
That row:
h1          5.000000
h2          1.503409
h3          2.285533
h4          2.441034
h5          4.919906
a           2.000000
b           3.627538
k           6.000000
S11       -28.420967
Metric     25.511194
gamma     281.396240
best      -28.420967
Name: (output/repeat_8, 44), dtype: float32
